In [28]:
pairs = [
    {
        "original": """
        Heat olive oil in a large skillet over medium heat. Add garlic and cook for 1–2 minutes until fragrant.
        Add tomatoes and simmer for 10 minutes. Boil pasta until al dente, then mix with sauce and serve.
        """,
        "simplified": """
        Heat oil in a pan.
        Add garlic and cook until fragrant.
        Add tomatoes and simmer.
        Cook pasta until al dente.
        Mix with sauce and serve.
        """
    }
]

In [29]:
def standardize_text(text):
    """
    Normalize text for consistent scoring.

    Strips extra whitespace and ensures the input is a string.
    Returns an empty string if the input is not a string.

    Parameters
    ----------
    text : any
        The text to standardize.

    Returns
    -------
    str
        A single-whitespace-separated string, or "" if input is invalid.
    """
    if not isinstance(text, str):
        return ""
    return ' '.join(text.split())

In [30]:
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer('all-MiniLM-L6-v2')

def compute_semantic_similarity(pair):
    """
    Compute cosine similarity between the original and simplified texts.

    Uses the all-MiniLM-L6-v2 sentence transformer to generate embeddings,
    then computes cosine similarity. A score of 1.0 means identical meaning;
    lower scores indicate semantic drift.

    Parameters
    ----------
    pair : dict
        A dictionary with keys 'original' and 'simplified', each containing
        a string of text.

    Returns
    -------
    float
        Cosine similarity score between 0.0 and 1.0.
    """
    original = pair['original']
    simplified = pair['simplified']

    original_std = standardize_text(original)
    simplified_std = standardize_text(simplified)

    embedding_original = model.encode(original_std, convert_to_tensor=True)
    embedding_simplified = model.encode(simplified_std, convert_to_tensor=True)

    similarity_score = util.cos_sim(embedding_original, embedding_simplified).item()

    return similarity_score

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
def compute_compression_ratio(pair):
    """
    Compute the word-count ratio of simplified text to original text.

    A ratio below 1.0 means the simplified text is shorter. A ratio of 0.5
    means the simplified text is half the length of the original.

    Parameters
    ----------
    pair : dict
        A dictionary with keys 'original' and 'simplified', each containing
        a string of text.

    Returns
    -------
    float
        Ratio of simplified word count to original word count.
        Returns 0.0 if the original text is empty.
    """
    original = pair['original']
    simplified = pair['simplified']

    original_std = standardize_text(original)
    simplified_std = standardize_text(simplified)

    original_word_count = len(original_std.split())
    simplified_word_count = len(simplified_std.split())

    if original_word_count == 0:
        return 0.0

    return simplified_word_count / original_word_count

In [32]:
import textstat

def compute_readability(pair):
    """
    Compute Flesch-Kincaid Grade Level for both original and simplified texts.

    A lower grade level indicates easier readability. Comparing the two scores
    shows whether simplification reduced the reading difficulty.

    Parameters
    ----------
    pair : dict
        A dictionary with keys 'original' and 'simplified', each containing
        a string of text.

    Returns
    -------
    tuple of (float, float)
        (readability_original, readability_simplified) — Flesch-Kincaid grade
        level for each text. Lower is easier to read.
    """
    original = pair['original']
    simplified = pair['simplified']

    original_std = standardize_text(original)
    simplified_std = standardize_text(simplified)

    readability_original = textstat.flesch_kincaid_grade(original_std)
    readability_simplified = textstat.flesch_kincaid_grade(simplified_std)

    return readability_original, readability_simplified

In [33]:
def evaluate_simplification(pair):
    """
    Run a full evaluation of a simplification pair and print a summary report.

    Combines semantic similarity, compression ratio, and readability improvement
    into a single overall score. Prints a formatted report and returns the raw
    metrics as a dictionary.

    Scoring rules:
        >= 0.80  Very Strong Simplification
        >= 0.65  Good
        >= 0.50  Moderate
        <  0.50  Weak

    Parameters
    ----------
    pair : dict
        A dictionary with keys 'original' and 'simplified', each containing
        a string of text.

    Returns
    -------
    dict
        Keys: similarity, compression, readability_original,
        readability_simplified, readability_improvement, overall_score, rating.
    """
    similarity = compute_semantic_similarity(pair)
    compression = compute_compression_ratio(pair)
    readability_original, readability_simplified = compute_readability(pair)

    readability_improvement = 1 - (readability_simplified / readability_original)
    overall_score = (similarity + compression + readability_improvement) / 3

    if overall_score >= 0.80:
        rating = "Very Strong Simplification"
    elif overall_score >= 0.65:
        rating = "Good"
    elif overall_score >= 0.50:
        rating = "Moderate"
    else:
        rating = "Weak"

    print("=" * 40)
    print("       SIMPLIFICATION EVALUATION")
    print("=" * 40)
    print(f"  Semantic Similarity:      {similarity:.4f}")
    print(f"  Compression Ratio:        {compression:.4f}")
    print(f"  Readability (Original):   {readability_original:.4f}")
    print(f"  Readability (Simplified): {readability_simplified:.4f}")
    print(f"  Readability Improvement:  {readability_improvement:.4f}")
    print("-" * 40)
    print(f"  Overall Score:            {overall_score:.4f}")
    print(f"  Rating:                   {rating}")
    print("=" * 40)

    return {
        "similarity": similarity,
        "compression": compression,
        "readability_original": readability_original,
        "readability_simplified": readability_simplified,
        "readability_improvement": readability_improvement,
        "overall_score": overall_score,
        "rating": rating
    }

In [34]:
import ssl                                                                                                     
import nltk
import os                                                                                                      
                  
ssl._create_default_https_context = ssl._create_unverified_context                                             
   
nltk_path = os.path.expanduser('~/nltk_data')                                                                  
result = nltk.download('cmudict', download_dir=nltk_path)
                                                                                                                 
                                                               
os.makedirs(nltk_path, exist_ok=True)
                                                                                                                              
print(result)   
print(nltk.data.find('corpora/cmudict'))  

True
/Users/nkemokweye/nltk_data/corpora/cmudict


[nltk_data] Downloading package cmudict to
[nltk_data]     /Users/nkemokweye/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!


In [35]:
evaluate_simplification(pairs[0]);

       SIMPLIFICATION EVALUATION
  Semantic Similarity:      0.8439
  Compression Ratio:        0.6757
  Readability (Original):   4.6013
  Readability (Simplified): 1.9360
  Readability Improvement:  0.5792
----------------------------------------
  Overall Score:            0.6996
  Rating:                   Good
